# 01 — Environment exploration (CarRacing-v3)

**Phase 1 exit criteria**
- action / observation spec (raw Gymnasium vs wrapped training pipeline),
- sample frames,
- random-policy reward histogram (20 episodes, single-env),
- avg **environment step time** & **episode wall-clock** (single-env).

In [ ]:
from __future__ import annotations

import time

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np

from racing_agent.env import make_car_racing_single

## 1. Action / observation specs

In [ ]:
raw = gym.make("CarRacing-v3", continuous=True, render_mode="rgb_array")
try:
    print("=== RAW CarRacing-v3 (inside gym.make wrappers) ===")
    print(raw.observation_space)
    print(raw.action_space)

    obs0, info0 = raw.reset(seed=0)
    print("initial obs shape", obs0.shape, obs0.dtype)
finally:
    raw.close()


wrapped_single = make_car_racing_single(
    seed=0,
    grayscale=True,
    resize_to=84,
    frame_stack=4,
    clip_reward=False,
    continuous=True,
    render_mode=None,
)

print()
print("=== WRAPPED pipeline (training) ===")
print(wrapped_single.observation_space)
print(wrapped_single.action_space)
obs1, info1 = wrapped_single.reset(seed=0)
print("stacked observation shape", obs1.shape, obs1.dtype)
wrapped_single.close()

**Note.** Grayscale uses `keep_dim=False` \((96,96)\); `FrameStack` yields **CxHxW** `(4, 84, 84)` uint8 (PyTorch-ready). We skip `VecTransposeImage`.

## 2. Visual comparison — RGB vs grayscale stack slices

In [ ]:
viz_raw = gym.make("CarRacing-v3", continuous=True, render_mode="rgb_array")
viz_wrap = make_car_racing_single(
    seed=1,
    grayscale=True,
    resize_to=84,
    frame_stack=4,
    clip_reward=False,
    continuous=True,
    render_mode="rgb_array",
)

try:
    rgb, _ = viz_raw.reset(seed=1)
    stacked, _ = viz_wrap.reset(seed=1)
    fig, ax = plt.subplots(1, 5, figsize=(14, 3))
    ax[0].imshow(rgb)
    ax[0].set_title("Raw RGB")
    ax[0].axis("off")
    for i in range(4):
        ax[i + 1].imshow(stacked[i], cmap="gray", vmin=0, vmax=255)
        ax[i + 1].set_title(f"Frame channel {i}")
        ax[i + 1].axis("off")
    plt.tight_layout()
    plt.show()
finally:
    viz_raw.close()
    viz_wrap.close()

## 3. Random policy — episode reward histogram (20 runs)

In [ ]:
N = 20
rewards: list[float] = []

for ep in range(N):
    env = make_car_racing_single(
        seed=100 + ep,
        grayscale=True,
        resize_to=84,
        frame_stack=4,
        clip_reward=False,
        continuous=True,
        render_mode=None,
    )
    try:
        _obs, _info = env.reset(seed=100 + ep)
        ep_rew = 0.0
        terminated = truncated = False
        while not (terminated or truncated):
            action = env.action_space.sample()
            _obs, r, terminated, truncated, _info = env.step(action)
            ep_rew += float(r)
        rewards.append(ep_rew)
    finally:
        env.close()


plt.figure(figsize=(8, 4))
plt.hist(rewards, bins=12, color="steelblue", edgecolor="black")
plt.xlabel("Episode return")
plt.ylabel("Count")
plt.title(f"Random policy — {N} episodes")
plt.axvline(float(np.mean(rewards)), color="orange", linestyle="--", label=f"mean={np.mean(rewards):.2f}")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print("mean", float(np.mean(rewards)), "std", float(np.std(rewards)), "min", float(np.min(rewards)), "max", float(np.max(rewards)))

## 4. Timing — average env step \& full episode duration

**Report requirement:** zmierz średni czas jednego kroku oraz jednego epizodu.

In [ ]:
bench = make_car_racing_single(
    seed=999,
    grayscale=True,
    resize_to=84,
    frame_stack=4,
    clip_reward=False,
    continuous=True,
    render_mode=None,
)

STEP_SAMPLES = 200
try:
    _obs, _info = bench.reset(seed=999)
    t0 = time.perf_counter()
    for _ in range(STEP_SAMPLES):
        bench.step(bench.action_space.sample())
    dt_batch = time.perf_counter() - t0
finally:
    bench.close()

episodes = 10
lengths: list[int] = []
ep_wall: list[float] = []

for ep in range(episodes):
        env = make_car_racing_single(
            seed=2000 + ep,
            grayscale=True,
            resize_to=84,
            frame_stack=4,
            clip_reward=False,
            continuous=True,
            render_mode=None,
        )
        try:
            _obs, _info = env.reset(seed=2000 + ep)
            t_ep = time.perf_counter()
            steps = 0
            terminated = truncated = False
            while not (terminated or truncated):
                env.step(env.action_space.sample())
                steps += 1
            ep_wall.append(time.perf_counter() - t_ep)
            lengths.append(steps)
        finally:
            env.close()


mean_step_ms = (dt_batch / STEP_SAMPLES) * 1000.0
mean_ep_s = float(np.mean(ep_wall))
mean_len = float(np.mean(lengths))

print(f"Warm env average step (~{STEP_SAMPLES} random steps):")
print(f"  per-step latency: {mean_step_ms:.3f} ms (wall-clock averaged)")
print()
print("Fresh env per episode (random policy):")
print(f"  avg episode wall-clock : {mean_ep_s:.3f} s")
print(f"  avg episode length      : {mean_len:.1f} env steps")

print()
print("(Per step within an episode ~= episode_time / episode_length):")
derived_ms = [(ep_wall[i] / max(lengths[i], 1)) * 1000.0 for i in range(episodes)]
print(f"  mean inferred step latency: {float(np.mean(derived_ms)):.3f} ms")

plt.figure(figsize=(8, 3))
plt.bar(["warm step batch", "ep / len (mean)"], [mean_step_ms, float(np.mean(derived_ms))], color=["steelblue", "orange"])
plt.ylabel("milliseconds")
plt.title("Timing comparison (illustrative; random actions)")
plt.grid(axis="y", alpha=0.3)
plt.show()